# 14 — Baseline Deep Dive: ISO 25010, HELM, MLflow

Detailed examination of the three baseline approaches, including their
adaptation methodology, metric coverage gaps, and raw scores by system.

Reference: `baselines/` directory — per-system evaluation JSONs,
methodology files, and monthly detection logs.

This complements notebook 04 (overall Wilcoxon comparison) by showing
*why* each baseline underperforms — their structural blind spots.


In [ ]:
import sys, os, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

SYSTEMS   = ['S1','S2','S3','S4']
BASELINES = ['iso25010','helm','mlflow']
DIMS      = ['FC','RO','SF','SS','TI','IQ']

# Load methodology files
methods = {}
for bl in BASELINES:
    key = bl if bl != 'iso25010' else 'iso25010'
    methods[bl] = json.load(open(f'../baselines/{bl}/{bl}_methodology.json'))

# Load per-system evaluations
evals = {bl: {} for bl in BASELINES}
for bl in BASELINES:
    for sid in SYSTEMS:
        fname = f'../baselines/{bl}/{sid}_{bl}_evaluation.json' if bl != 'mlflow' else f'../baselines/{bl}/{sid}_mlflow_runs.json'
        evals[bl][sid] = json.load(open(fname))

print("Baselines loaded:")
for bl, m in methods.items():
    full_name = m.get('full_name', bl)
    ref = m.get('reference','')
    print(f"  {bl}: {full_name}  ({ref})")

Baselines loaded:
  iso25010: ISO/IEC 25010:2023 (adapted)  (ISO/IEC JTC 1/SC 7 2023)
  helm:     HELM (Holistic Evaluation of Language Models)  (Liang et al. 2023)
  mlflow:   MLflow (adapted for LLM quality monitoring)  (Zaharia et al. 2018)


## 1. Methodology overview — adaptation approach

In [ ]:
for bl, m in methods.items():
    print(f"{'='*55}")
    print(f"  {m.get('full_name', bl)}")
    print(f"{'='*55}")
    print(f"  Evaluation unit:   {m.get('evaluation_unit','N/A')}")
    print(f"  Scenarios applied: {m.get('n_scenarios_applied','N/A')} / {m.get('n_scenarios_full','N/A')} total")
    lims = m.get('key_limitations', [])
    print(f"  Key limitations:")
    for lim in lims[:3]:
        print(f"    • {lim}")
    print()

  ISO/IEC 25010:2023 (adapted)
  Evaluation unit:   characteristic/sub-characteristic
  Scenarios applied: 8 / 31 total
  Key limitations:
    • No hallucination-specific metric — semantic faithfulness not covered
    • Transparency limited to documentation quality, not runtime explainability
    • Thresholds not operationalised — qualitative assessment only

  HELM (Holistic Evaluation of Language Models)
  Evaluation unit:   benchmark scenario
  Scenarios applied: 6 / 42 total
  Key limitations:
    • Model-centric — evaluates model capability, not deployed system quality
    • No integration quality (IQ) coverage — infrastructure entirely absent
    • Static benchmark — cannot detect production drift or regression

  MLflow (adapted for LLM quality monitoring)
  Evaluation unit:   run/experiment
  Scenarios applied: 9 / 18 total
  Key limitations:
    • Designed for ML experiments, not production LLM quality assurance
    • No semantic faithfulness or transparency metrics
    • Aler

## 2. Dimension coverage comparison

In [ ]:
# Coverage scores from comparative_analysis_full.csv (mean across all 4 systems)
comp_df = pd.read_csv('../baselines/comparative_analysis_full.csv')
cov = comp_df.groupby(['dimension','approach'])['coverage_score'].mean().unstack()
approaches_order = ['QALIS','ISO25010','HELM','MLflow']
cov = cov[approaches_order]

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(DIMS))
w = 0.2
colors = {'QALIS':'#2ecc71','ISO25010':'#e74c3c','HELM':'#3498db','MLflow':'#f39c12'}

for i, ap in enumerate(approaches_order):
    vals = [cov.loc[d, ap] if d in cov.index and ap in cov.columns else 0 for d in DIMS]
    ax.bar(x + i*w, vals, w, label=ap, color=colors[ap], alpha=0.85, edgecolor='white')

ax.set_xticks(x + 1.5*w); ax.set_xticklabels(DIMS, fontsize=11)
ax.set_ylim(0, 1.05); ax.set_ylabel('Coverage Score (0–1)')
ax.set_title('Dimension Coverage by Approach  (mean across S1–S4)', fontsize=12)
ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("\nBlind spot summary (coverage < 0.40):")
for ap in approaches_order[1:]:
    gaps = [d for d in DIMS if cov.loc[d, ap] < 0.40]
    if gaps:
        print(f"  {ap}: {', '.join(gaps)}")


Blind spot summary (coverage < 0.40):
  ISO25010: SF, TI
  HELM: IQ, TI
  MLflow: SF, TI


## 3. Monthly detection logs — flat baselines vs QALIS improvement

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for bl in BASELINES:
    log = pd.read_csv(f'../baselines/{bl}/{bl}_monthly_detection_log.csv')
    monthly = log.groupby('month')['detection_rate'].mean()
    label = methods[bl]['full_name'].split('(')[0].strip()
    ax.plot([1,2,3], monthly.values[:3], 'o--', color=colors[bl.upper() if bl!='iso25010' else 'ISO25010'],
            label=label, linewidth=1.8, markersize=7, alpha=0.85)

# QALIS from longitudinal data
qalis_long = pd.read_csv('../data/processed/longitudinal/defect_detection_longitudinal.csv')
qalis_monthly = qalis_long[qalis_long['approach']=='QALIS'].groupby('month')['detection_rate'].mean()
ax.plot([1,2,3], qalis_monthly.values[:3], 'o-', color=colors['QALIS'],
        label='QALIS', linewidth=2.5, markersize=9)

ax.set_xticks([1,2,3]); ax.set_xticklabels(['Oct 2024','Nov 2024','Dec 2024'])
ax.set_ylabel('Mean Detection Rate'); ax.set_ylim(0.3, 1.0)
ax.set_title('Monthly Detection Rate — All Approaches\n(QALIS improves; baselines flat)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Trend test for each baseline
from scipy.stats import linregress
print("\nLinear trend (slope ≈ 0 confirms flat baseline):")
for bl in BASELINES:
    log = pd.read_csv(f'../baselines/{bl}/{bl}_monthly_detection_log.csv')
    vals = log.groupby('month')['detection_rate'].mean().values[:3]
    slope, _, _, p, _ = linregress([1,2,3], vals)
    sig = "p<0.05" if p < 0.05 else "n.s."
    print(f"  {bl:<10}: slope={slope:+.4f}  {sig}")


Linear trend (slope ≈ 0 confirms flat baseline):
  iso25010  : slope=+0.0110  n.s.
  helm      : slope=+0.0065  n.s.
  mlflow    : slope=+0.0090  n.s.


## 4. Per-system baseline evaluation summary

In [ ]:
print("Baseline effectiveness scores by system and dimension:\n")
for bl in BASELINES:
    bl_name = methods[bl]['full_name'].split('(')[0].strip()
    print(f"  ── {bl_name} ──")
    for sid in SYSTEMS:
        ev = evals[bl][sid]
        dim_scores = ev.get('dimension_coverage', ev.get('dimension_scores', {}))
        scores_str = "  ".join(f"{d}={dim_scores.get(d, 0):.2f}" for d in DIMS)
        print(f"    {sid}: {scores_str}")
    print()

Baseline effectiveness scores by system and dimension:

  ── ISO/IEC 25010:2023 (adapted) ──
    S1: FC=0.69  RO=0.54  SF=0.31  SS=0.58  TI=0.29  IQ=0.71
    S2: FC=0.71  RO=0.57  SF=0.30  SS=0.62  TI=0.31  IQ=0.73
    S3: FC=0.65  RO=0.52  SF=0.32  SS=0.55  TI=0.28  IQ=0.70
    S4: FC=0.68  RO=0.56  SF=0.29  SS=0.60  TI=0.30  IQ=0.70

  ── HELM (Holistic Evaluation of Language Models) ──
    S1: FC=0.76  RO=0.70  SF=0.61  SS=0.49  TI=0.38  IQ=0.21
    S2: FC=0.78  RO=0.72  SF=0.63  SS=0.52  TI=0.40  IQ=0.23
    S3: FC=0.72  RO=0.67  SF=0.62  SS=0.48  TI=0.37  IQ=0.22
    S4: FC=0.74  RO=0.70  SF=0.60  SS=0.50  TI=0.39  IQ=0.22

  ── MLflow (adapted for LLM quality monitoring) ──
    S1: FC=0.52  RO=0.44  SF=0.28  SS=0.61  TI=0.19  IQ=0.77
    S2: FC=0.55  RO=0.46  SF=0.29  SS=0.65  TI=0.21  IQ=0.79
    S3: FC=0.50  RO=0.42  SF=0.27  SS=0.59  TI=0.18  IQ=0.76
    S4: FC=0.53  RO=0.44  SF=0.28  SS=0.62  TI=0.20  IQ=0.77
